In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df=pd.read_excel('bank_data.xlsx')
df

,CustomerID,Age,Gender,Income,Balance,Transactions,CreditScore,Loan,Region,Tenure,IsActive
0,1,44,Male,2198,25518.0,61,NaN,No,South,3,No
1,2,30,Female,5158,28068.0,96,841.0,No,East,7,Yes
2,3,20,Female,747,43922.0,10,613.0,No,South,9,Yes
3,4,28,Male,2279,47635.0,53,NaN,Yes,North,13,Yes
4,5,61,Female,2360,25105.0,102,830.0,No,West,9,Yes
...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,35,Female,5708,36671.0,51,777.0,Yes,East,17,No
9996,9997,34,Male,9162,37419.0,68,713.0,Yes,West,2,Yes
9997,9998,34,Female,8321,41633.0,80,308.0,No,North,10,Yes
9998,9999,28,Male,6228,28653.0,134,899.0,Yes,South,11,No


In [56]:
# We get information about the dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   CustomerID    10000 non-null  int64  
 1   Age           10000 non-null  int64  
 2   Gender        10000 non-null  str    
 3   Income        10000 non-null  int64  
 4   Balance       10000 non-null  float64
 5   Transactions  10000 non-null  int64  
 6   CreditScore   9000 non-null   float64
 7   Loan          10000 non-null  str    
 8   Region        10000 non-null  str    
 9   Tenure        10000 non-null  int64  
 10  IsActive      10000 non-null  str    
dtypes: float64(2), int64(5), str(4)
memory usage: 859.5 KB


In [57]:
# Count Rows 
df.shape[0]

10000

In [58]:
# Data Cleaning
# CreditScore missing values fill
df["CreditScore"]=df["CreditScore"].fillna(df["CreditScore"].median())
# Balance outlier removal(IQR)
Q1=df['Balance'].quantile(0.25)
Q3=df['Balance'].quantile(0.75)
IQR=Q3-Q1
loweroutlier=Q1-1.5*IQR
upperoutlier=Q3+1.5*IQR
data=df[(df['Balance']>=loweroutlier)&(df['Balance']<=upperoutlier)]

In [59]:
data.info()

<class 'pandas.DataFrame'>
Index: 9500 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   CustomerID    9500 non-null   int64  
 1   Age           9500 non-null   int64  
 2   Gender        9500 non-null   str    
 3   Income        9500 non-null   int64  
 4   Balance       9500 non-null   float64
 5   Transactions  9500 non-null   int64  
 6   CreditScore   9500 non-null   float64
 7   Loan          9500 non-null   str    
 8   Region        9500 non-null   str    
 9   Tenure        9500 non-null   int64  
 10  IsActive      9500 non-null   str    
dtypes: float64(2), int64(5), str(4)
memory usage: 890.6 KB


In [60]:
# Average Age
int(data['Age'].mean())

46

In [61]:
# Gender counts
data['Gender'].value_counts()

Gender
Female    4777
Male      4723
Name: count, dtype: int64

In [62]:
# Number of customers by region
data['Region'].value_counts()

Region
West     2450
North    2370
South    2340
East     2340
Name: count, dtype: int64

In [63]:
# Average Balance
round(data['Balance'].mean(),3)

np.float64(24785.175)

In [64]:
# Active vs Inactive
data['IsActive'].value_counts()

IsActive
Yes    4774
No     4726
Name: count, dtype: int64

In [65]:
# Top 10 customers with the highest balance
data.sort_values(by='Balance',ascending=False).head(10)

,CustomerID,Age,Gender,Income,Balance,Transactions,CreditScore,Loan,Region,Tenure,IsActive
3120,3121,22,Female,3541,49996.0,30,419.0,Yes,East,12,Yes
766,767,29,Female,3796,49995.0,157,427.0,No,East,15,No
2362,2363,43,Female,8904,49987.0,172,787.0,Yes,South,14,No
2355,2356,51,Female,7355,49982.0,200,676.0,No,South,9,No
2840,2841,38,Male,7341,49976.0,87,605.0,Yes,West,11,No
335,336,41,Female,5503,49972.0,77,598.0,Yes,East,3,Yes
2094,2095,30,Female,2931,49969.0,165,796.0,No,East,18,Yes
4087,4088,34,Male,1117,49969.0,32,853.0,No,North,17,Yes
9360,9361,70,Male,1742,49955.0,164,482.0,Yes,East,18,Yes
3385,3386,45,Female,8016,49952.0,170,781.0,Yes,West,3,No


In [66]:
# Average age by gender and activity status
data.groupby(['Gender','IsActive'])["Age"].mean()

Gender  IsActive
Female  No          47.031884
        Yes         46.556732
Male    No          46.264388
        Yes         46.390962
Name: Age, dtype: float64

In [67]:
# Average age by Region
data.groupby("Region")["Balance"].mean()

Region
East     25095.880769
North    24578.822363
South    24679.938462
West     24788.545714
Name: Balance, dtype: float64

In [68]:
# Average age by Region and Gender
data.groupby(["Region","Gender"])["Balance"].mean()

Region  Gender
East    Female    25237.394281
        Male      24958.185497
North   Female    24476.630952
        Male      24679.473199
South   Female    24551.999171
        Male      24816.233892
West    Female    23946.211290
        Male      25651.764463
Name: Balance, dtype: float64

In [69]:
# Income quartile analysis
data["Income"].quantile([0.25,0.5,0.75])

0.25    2892.00
0.50    5330.00
0.75    7669.25
Name: Income, dtype: float64

In [70]:
# Customer segmentation
def segment(income):
    if income<2000:
        return 'Low'
    elif income<=5000:
        return 'Middle'
    else:
        return 'High'
data['Segment']=data['Income'].apply(segment)
data['Segment'].value_counts()

Segment
High      5090
Middle    2957
Low       1453
Name: count, dtype: int64

In [ ]:
# Suspicious activity
suspicious=data[(data['Transactions']>100) & (data['Balance']<1000)]
suspicious

,CustomerID,Age,Gender,Income,Balance,Transactions,CreditScore,Loan,Region,Tenure,IsActive,Segment
101,102,36,Female,5138,882.0,108,598.0,Yes,North,11,Yes,High
107,108,32,Female,5308,602.0,185,598.0,No,North,4,No,High
147,148,43,Female,8427,267.0,132,893.0,Yes,North,19,No,High
260,261,67,Female,4341,976.0,114,364.0,No,West,19,No,Middle
288,289,72,Female,4827,779.0,181,438.0,No,West,7,Yes,Middle
...,...,...,...,...,...,...,...,...,...,...,...,...
9738,9739,59,Male,8060,807.0,192,894.0,No,East,20,Yes,High
9770,9771,74,Female,8833,75.0,181,352.0,No,North,20,No,High
9775,9776,60,Male,8169,689.0,142,816.0,No,East,13,No,High
9811,9812,61,Female,1476,474.0,189,393.0,No,South,6,Yes,Low


In [83]:
# Shows the percentage of people with a loan.
loanrate=(data['Loan']=='Yes').mean()*100
print(f"{loanrate:.2f}%")

50.12%


In [ ]:
# Income Range
bins=[0, 2000, 4000, 6000, 10000]
labels=['0-2K', '2K-4K', '4K-6K','6K-10K']
data['IncomeRange']=pd.cut(data['Income'],bins=bins,labels=labels)
result=data['IncomeRange'].value_counts()
print(result)
data

IncomeRange
6K-10K    4060
4K-6K     2003
2K-4K     1982
0-2K      1455
Name: count, dtype: int64


,CustomerID,Age,Gender,Income,Balance,Transactions,CreditScore,Loan,Region,Tenure,IsActive,Segment,IncomeRange
0,1,44,Male,2198,25518.0,61,598.0,No,South,3,No,Middle,2K-4K
1,2,30,Female,5158,28068.0,96,841.0,No,East,7,Yes,High,4K-6K
2,3,20,Female,747,43922.0,10,613.0,No,South,9,Yes,Low,0-2K
3,4,28,Male,2279,47635.0,53,598.0,Yes,North,13,Yes,Middle,2K-4K
4,5,61,Female,2360,25105.0,102,830.0,No,West,9,Yes,Middle,2K-4K
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,35,Female,5708,36671.0,51,777.0,Yes,East,17,No,High,4K-6K
9996,9997,34,Male,9162,37419.0,68,713.0,Yes,West,2,Yes,High,6K-10K
9997,9998,34,Female,8321,41633.0,80,308.0,No,North,10,Yes,High,6K-10K
9998,9999,28,Male,6228,28653.0,134,899.0,Yes,South,11,No,High,6K-10K


In [ ]:
# Detect anomalies: Customer with low balance but high transactions or high balance but low transactions 